In [3]:
import xgboost as xgb
print(xgb.__version__)

3.0.4


In [4]:
import sys, xgboost as xgb
print(sys.executable)        # should point to .../.venv/bin/python
print(xgb.__version__)       # should print 3.0.4
print(xgb.__file__)          # should live under .../.venv/...

c:\Users\saada\.conda\envs\ml_env\python.exe
3.0.4
c:\Users\saada\.conda\envs\ml_env\Lib\site-packages\xgboost\__init__.py


In [5]:
# ==============================================
# 1. Imports
# ==============================================
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor
import optuna
import mlflow
import mlflow.xgboost

c:\Users\saada\.conda\envs\ml_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# ==============================================
# 2. Load processed datasets
# ==============================================
train_df = pd.read_csv(r"D:\REGRESSION_ML_ENDTOEND\Regression_ML_EndtoEnd\data\processed\feature_engineered_train.csv")
eval_df  = pd.read_csv(r"D:\REGRESSION_ML_ENDTOEND\Regression_ML_EndtoEnd\data\processed\feature_engineered_eval.csv")


# Define target + features
target = "price"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_eval, y_eval   = eval_df.drop(columns=[target]), eval_df[target]

print("Train shape:", X_train.shape)
print("Eval shape:", X_eval.shape)

Train shape: (576815, 39)
Eval shape: (148448, 39)


In [7]:
# ==============================================
# 3. Define Optuna objective function with MLflow
# ==============================================
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
        "tree_method": "hist",
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_eval)
        rmse = float(np.sqrt(mean_squared_error(y_eval, y_pred)))
        mae = float(mean_absolute_error(y_eval, y_pred))
        r2 = float(r2_score(y_eval, y_pred))

        # Log hyperparameters + metrics
        mlflow.log_params(params)
        mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})

    return rmse

In [8]:
# ==============================================
# 4. Run Optuna study with MLflow
# ==============================================
# Force MLflow to always use the root project mlruns folder
# Use a relative path so it creates 'mlruns' right next to your notebook
# Use the file:/// prefix for local Windows paths
mlflow.set_tracking_uri(r"file:///D:/REGRESSION_ML_ENDTOEND/Regression_ML_EndtoEnd/mlruns")
mlflow.set_experiment("xgboost_optuna_housing")

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

c:\Users\saada\.conda\envs\ml_env\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/03/24 15:02:47 INFO mlflow.tracking.fluent: Experiment with name 'xgboost_optuna_housing' does not exist. Creating a new experiment.


[I 2026-03-24 15:02:47,464] A new study created in memory with name: no-name-89f789c2-8dc8-4a4e-92a8-6806131e8eb5
[I 2026-03-24 15:03:18,763] Trial 0 finished with value: 79137.40012097714 and parameters: {'n_estimators': 200, 'max_depth': 5, 'learning_rate': 0.0553720226101658, 'subsample': 0.5868262716927544, 'colsample_bytree': 0.8940616791668899, 'min_child_weight': 3, 'gamma': 2.4135029689471725, 'reg_alpha': 1.1894540368885612e-05, 'reg_lambda': 1.875660501002429e-08}. Best is trial 0 with value: 79137.40012097714.
[I 2026-03-24 15:03:51,213] Trial 1 finished with value: 70984.22656166885 and parameters: {'n_estimators': 513, 'max_depth': 5, 'learning_rate': 0.10793172489996654, 'subsample': 0.7175648693454404, 'colsample_bytree': 0.5125225518535593, 'min_child_weight': 7, 'gamma': 1.9171686051271208, 'reg_alpha': 0.00016841029856741243, 'reg_lambda': 7.160827910451412e-08}. Best is trial 1 with value: 70984.22656166885.
[I 2026-03-24 15:04:08,648] Trial 2 finished with value: 92

Best params: {'n_estimators': 995, 'max_depth': 8, 'learning_rate': 0.03665165259633451, 'subsample': 0.8569129710807791, 'colsample_bytree': 0.5912722812808898, 'min_child_weight': 10, 'gamma': 1.2037476463885404, 'reg_alpha': 7.369882679872359, 'reg_lambda': 6.3880827897492285}


In [9]:
# ==============================================
# 5. Train final model with best params and log to MLflow
# ==============================================
best_params = study.best_trial.params
best_model = XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_eval)

mae = mean_absolute_error(y_eval, y_pred)
rmse = np.sqrt(mean_squared_error(y_eval, y_pred))
r2 = r2_score(y_eval, y_pred)

print("Final tuned model performance:")
print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)


# Log final model
with mlflow.start_run(run_name="best_xgboost_model"):
    mlflow.log_params(best_params)
    mlflow.log_metrics({"rmse": rmse, "mae": mae, "r2": r2})
    
    # Use the sklearn flavor for better stability with XGBRegressor
    # Note: Use 'name' instead of 'artifact_path' as per the MLflow warning
    mlflow.sklearn.log_model(
        sk_model=best_model, 
        name="housing_xgboost_model"
    )

Final tuned model performance:
MAE: 30714.716273933558
RMSE: 69702.55938577784
R²: 0.962454535521192


2026/03/24 15:16:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
